In [ ]:
# Import packages
import numpy as np
import pandas as pd
import random as random
import matplotlib.pyplot as plt
import scipy

# from tqdm import tqdm
import seaborn as sns

# load in bettor objects

from bettor import bettor, run_market, run_market_lob, plot_returns, run_market_fixed_el, gen_election

In [ ]:
from statsmodels.tsa.stattools import grangercausalitytests

In [ ]:
from matplotlib.colors import LinearSegmentedColormap

In [ ]:
colors = [
    "#CC99FF",  # lavender purple (existing)
    "#000080",  # navy blue (existing)
    "#339966",  # forest green (existing)
    "#F4A261",  # muted orange
    "#B22222",  # deep ruby red
    "#6A4C93",  # royal violet
    "#1D3557",  # dark indigo
    "#2A9D8F",  # teal
    "#457B9D",   # steel blue,
    "#FFB347",  # warm amber/orange
    "#E63946"  # deep crimson red
]

# define mpl colourmap
cmap= LinearSegmentedColormap.from_list("power_predictioncmap", colors)

In [ ]:
from collections.abc import Iterable

# need a function to flatten irregular list of lists
def flatten(xs):
    for x in xs:
        if isinstance(x, Iterable) and not isinstance(x, (str, bytes)):
            yield from flatten(x)
        else:
            yield x

In [ ]:
def evaluate_markets(market_record, maxlag=5):
    mp = np.array(market_record['price_history'])
    tp = np.array(market_record['gen_el'])

    # take first differences
    tp_diff = np.diff(tp)
    mp_diff = np.diff(mp)
    ts_data = np.column_stack([mp_diff, tp_diff])

    mse = np.mean((mp-tp)**2)

    res = grangercausalitytests(ts_data, maxlag=maxlag, addconst=True, verbose=False)

    # pull p-values from one of the tests at each lag (e.g., ssr_ftest)
    pvals = {lag: res[lag][0]['ssr_ftest'][1] for lag in res}

    # decide significance at, say, 5%
    sig_lags = [lag for lag, p in pvals.items() if p < 0.05]
    # best_lag = min(sig_lags) if sig_lags else None

    # get lag with lowest p-value
    if pvals:
        best_lag = min(pvals, key=pvals.get)
    else:
        best_lag = None

    return mse, best_lag

In [ ]:
# Global parameters

N_bettors = 100
T_max = 500
initial_price = 0.5
rw_variance = 0.05

### Market mechanism

`MECHANISM = 'lob'` runs the central limit order book (`run_market_lob`): bettors arrive in random order, post limit orders, orders match by price-time priority, and the recorded price is the end-of-period mid-quote. `MECHANISM = 'walrasian'` runs the original `run_market` (`set_market_price` with `market_lambda`), so both can be compared with the same notebook.

Note: in the original `run_market`, portfolios are only updated for the first `n_bettors` agents, so whales appended to the list place orders but never pay for them (effectively unlimited budget). In `run_market_lob` every agent in the list trades and settles, so whales are budget-constrained.

In [ ]:
# Market mechanism settings
MECHANISM = 'lob'          # 'lob' = central limit order book, 'walrasian' = original set_market_price

lob_params = {'tick': 0.01,          # price grid (1 cent)
              'max_markup': 0.05,    # max shading of limit price away from own valuation (drives spread)
              'ttl': 5,              # periods an unfilled order stays in the book
              'price_rule': 'mid'}   # recorded price: 'mid' (mid-quote) or 'last' (last trade)

RESET_EACH_RUN = False     # False reproduces the original notebook: the same bettor objects carry their
                           # budgets, holdings and beliefs from one iteration into the next.
                           # True draws a fresh population for every iteration (independent replications).

SEED = 42
rng = np.random.default_rng(SEED)
np.random.seed(SEED)

def run_one(params):
    '''Run one market with the chosen mechanism; returns the usual market_record dict.'''
    if MECHANISM == 'lob':
        return run_market_lob(**params, **lob_params, rng=rng)
    return run_market(**params)

suffix = '_lob' if MECHANISM == 'lob' else ''

### Single-run check

One market with the whale (valuation 0.7) to check the mechanism before the full loop: price vs. true probability, and (order book only) the bid-ask spread and trades per period.

In [ ]:
# same set-up as the main loop below: 100 bettors with budget 100, one whale holding half of all money
check_bettors = [bettor(budget=100, expertise=0.9) for i in range(N_bettors)] + \
                [bettor(budget=100*N_bettors, whale=True, market_valuation=0.7)]
check_rec = run_one({'n_bettors': N_bettors, 't_election': T_max, 'initial_price': initial_price,
                     'outcome_uncertainty': rw_variance, 'bettors': check_bettors})

n_panels = 3 if MECHANISM == 'lob' else 1
fig, ax = plt.subplots(n_panels, 1, figsize=[9, 2.6*n_panels + 1], sharex=True, squeeze=False)
ax = ax[:, 0]
ax[0].plot(check_rec['gen_el'], color='#B22222', label='True probability')
ax[0].plot(check_rec['price_history'], color='black', label='Market price')
ax[0].plot(check_rec['beliefs'], color='#6A4C93', linestyle='dotted', label='Mean belief')
ax[0].set_ylim(0, 1.02); ax[0].set_ylabel('Price'); ax[0].legend(loc='lower left', fontsize=8)
if MECHANISM == 'lob':
    ax[1].plot(check_rec['spread'], color='#457B9D'); ax[1].set_ylabel('Bid-ask spread')
    ax[2].plot(check_rec['n_trades'], color='#339966'); ax[2].set_ylabel('Trades per period')
ax[-1].set_xlabel('Period')
fig.tight_layout()

### Market shock

In [ ]:
n_iter_=100

In [ ]:
# Set initial input values to the betting market function
parameters = {'n_bettors': N_bettors, # The number of betting agents
                #'el_outcome': 1, # Q: Ultimate election outcome - assuming we know this to begin with and it does not change over time...for now this is implemented as a random walk of the probability...but should this be 0 or 1 instead? '''
            't_election': T_max, # Time until election takes place (ie. time horizon of betting)
            'initial_price': initial_price, # Initial market price (is this equivalent to probability of winning)
            'outcome_uncertainty': rw_variance} # This is a measure of how uncertain the true outcome is - ie. the volatility of the random walk election probability


# expertise_values = np.linspace(0,1,N_bettors)

av_budget=100
budget_total = av_budget* N_bettors
N_whales = 1
whale_valuation = 0.6

prop_whale=0.5

expertise_whale = {}
lob_diagnostics = {w: [] for w in [0.6, 0.7, 0.8]}

for whale_valuation in [0.6,0.7,0.8]:
    
    whale_effect = []

    whale_budget = av_budget*N_bettors*prop_whale / (1-prop_whale)

    def make_population():
        non_whale_bettors = [bettor(budget=av_budget, expertise=0.9) for i in range(N_bettors)]
        whale_bettors = [bettor(budget=whale_budget, whale=True, market_valuation=whale_valuation) for _ in range(N_whales)]
        return non_whale_bettors + whale_bettors

    parameters.update({'bettors': make_population()})

    for _ in range(n_iter_):
        if RESET_EACH_RUN:
            parameters.update({'bettors': make_population()})
        market_record = run_one(parameters)
        lob_diagnostics[whale_valuation].append({'spread': market_record.get('spread'),
                                                 'n_trades': market_record.get('n_trades')})

        whale_effect.append((np.array(market_record['price_history']) -  np.array(market_record['gen_el'])))

    expertise_whale[whale_valuation] = whale_effect

In [ ]:
times = np.arange(T_max+1)
vals = []
for k,val in expertise_whale.items():
    for v in val:
        for x in range(T_max+1):
            vals.append([k,v[x],times[x]])


df_plot = pd.DataFrame(vals, columns = ['W','SE','T'])

In [ ]:
plt.figure(figsize=[10, 5])

sns.lineplot(data = df_plot, x='T', y='SE',hue='W', palette = ["#457B9D", "#339966", "#B22222"], linewidth=1.5)
plt.xlabel('Time since shock')

plt.ylabel('Absolute difference between market price and \n true election outcome')

# clip axhlines to max at 500
plt.axhline(0.2, color='black', linestyle='--', alpha=0.8)
plt.axhline(0.1, color='black', linestyle='--', alpha=0.8)
plt.axhline(0.3, color='black', linestyle='--', label='Theoretical limits', alpha=0.8)

# round labels in legend to 1dp
# get current legend
legend = plt.gca().get_legend()
for text in legend.get_texts():
    text.set_text(f"{float(text.get_text()):.1f}")

# move legend
plt.ylim(0,0.55)
plt.legend(title = 'Valuation of Whale', loc='upper right')

plt.savefig(f'TheoreticalAgreement{suffix}.pdf', bbox_inches='tight')

In [ ]:
# Order-book diagnostics averaged over runs (LOB only)
if MECHANISM == 'lob':
    for w, runs in lob_diagnostics.items():
        print(f"whale valuation {w}: mean spread {np.nanmean([np.nanmean(r['spread']) for r in runs]):.3f}, "
              f"mean trades/period {np.mean([np.mean(r['n_trades']) for r in runs]):.1f}")